# Lab 5 — Searching Quora Questions

**MA553 · Session 6**

Name:

---

In this lab you build the search from the session and measure it.
You turn 522,931 Quora questions into vectors, search them for 1,000 query
questions, and count how many marked duplicates the search finds.

**Part A** builds the search and measures it with recall at 10.

**Part B** compares it with keyword search, with shorter vectors and with an
approximate index, and looks at what the measurement misses.

Work through the cells in order. Six cells are marked **Write this yourself**:
A4, A5, A6, B1, B2 and B3. Each says which variables to create. Later cells use
those names. The **Answer here** cells are short written answers.

# Part A — Build and measure the search

## Setup

This lab uses the Quora questions and the embedding model downloaded by
`get_data.py`. If the folders `data/quora` and `model/all-MiniLM-L6-v2` are
missing, run `python get_data.py` in this folder first.

Run this.

In [ ]:
import json
import time
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

DATA = Path("data/quora")
model = SentenceTransformer("model/all-MiniLM-L6-v2")
print("Model loaded.")

## A1 — The questions, the queries and the marked duplicates

This cell loads three things:

- `texts`: the 522,931 questions of the collection, and `ids`, their ids.
- `query_text`: the text of each query question, by id.
- `marked`: for each query id, the set of question ids that people marked as
  duplicates.

It then draws the same 1,000 queries as the slides, into the list `queries`.

Run this.

In [ ]:
corpus = [json.loads(line) for line in open(DATA / "corpus.jsonl")]
ids = np.array([c["_id"] for c in corpus])
texts = [c["text"] for c in corpus]
query_text = {q["_id"]: q["text"] for q in map(json.loads, open(DATA / "queries.jsonl"))}

marked = {}
for line in list(open(DATA / "qrels" / "test.tsv"))[1:]:
    query_id, question_id, _ = line.split()
    marked.setdefault(query_id, set()).add(question_id)

rng = np.random.default_rng(6)
queries = rng.choice(sorted(marked), 1000, replace=False).tolist()

print(f"Questions in the collection: {len(texts):,}")
print(f"Queries: {len(queries):,}")
q = queries[0]
print("\nExample query:", query_text[q])
print("Its marked duplicates:", [texts[i] for i in np.flatnonzero(np.isin(ids, list(marked[q])))])

## A2 — Embed the collection

This cell turns every question into a vector of 384 numbers and saves the
vectors in `vectors.npy`. If the file already exists, the cell loads it.

The *length* of a vector is the square root of the sum of its squared numbers.
The vectors have length 1, so the cosine similarity of two vectors is their
dot product.

**Running time.** Up to about an hour on a laptop without a GPU, and a few
minutes with a GPU (for example, a Colab GPU runtime). Run this once: the
vectors are saved, and later runs load them in seconds.

In [ ]:
if Path("vectors.npy").exists():
    X = np.load("vectors.npy")
else:
    X = model.encode(texts, batch_size=256, normalize_embeddings=True,
                     show_progress_bar=True).astype("float32")
    np.save("vectors.npy", X)

Q = model.encode([query_text[q] for q in queries], normalize_embeddings=True).astype("float32")
print("Collection vectors:", X.shape)
print("Query vectors:     ", Q.shape)

## A3 — Try the search

The function `search(text, k)` embeds any text, computes its cosine
similarity with every question, and prints the top k questions.

Run the examples. Then try two questions of your own in the last line.

In [ ]:
def search(text, k=5):
    v = model.encode([text], normalize_embeddings=True)[0]
    cos = X @ v
    top = np.argsort(-cos)[:k]
    print(text)
    for i in top:
        print(f"  {cos[i]:.2f}  {texts[i]}")
    print()

search("How do I start a property management firm?")
search("Is it safe to travel to India?")
search("Comment apprendre le Python rapidement?")
# search("your own question here")

**Answer here.** Which of your own questions gave the best top 5, and which
gave the worst? For the French question, what do the results have in common,
and why?

*Answer here.*

## A4 — Cosine similarity by hand ✏️ Write this yourself

**Goal.** Check that the cosine similarity of two vectors is their dot product
when both vectors have length 1.

1. Embed the two questions below with `model.encode(..., normalize_embeddings=True)`.
2. Compute the cosine similarity with the formula
   (dot product) ÷ (length of a × length of b). Use `np.dot` and
   `np.linalg.norm`. Store it in `cos_formula`.
3. Compute the plain dot product and store it in `cos_dot`.
4. Print both.

**Expected.** Both are 0.98.

In [ ]:
# Your code here.

## The top 10 for every query

The function `top10(collection, query_vectors)` returns, for each query, the
ids of its 10 nearest questions. It compares every query with every question
(exact search). Part B uses it again with other vectors.

Run this. It takes under a minute.

In [ ]:
def top10(collection, query_vectors):
    results = []
    for start in range(0, len(query_vectors), 100):
        scores = query_vectors[start:start + 100] @ collection.T
        best = np.argpartition(-scores, 10, axis=1)[:, :10]
        for row, cand in zip(scores, best):
            results.append(ids[cand[np.argsort(-row[cand])]].tolist())
    return results

found = top10(X, Q)
print("Top 10 for the first query:", found[0])

## A5 — Recall at 10 ✏️ Write this yourself

**Goal.** Measure how many marked duplicates the search finds.

For one query, recall at 10 is

(number of marked duplicates in the top 10) ÷ (number of marked duplicates).

1. Write a function `recall_at_10(found)` that takes the list of top-10 lists
   (in the same order as `queries`) and returns a NumPy array with one recall
   value per query. Use `marked[q]` for the marked duplicates of query `q`.
2. Store the result for `found` in `recall`, and print its mean.
3. Print how many queries have recall 1 and how many have recall 0.

**Expected.** Mean recall at 10 of 0.96. 932 queries with recall 1, and
21 with recall 0.

In [ ]:
# Your code here.

## A6 — An interval for recall at 10 ✏️ Write this yourself

**Goal.** Measure how far the mean recall could move with a different draw of
1,000 queries. Use the bootstrap of Session 5.

1. Draw 1,000 values with replacement from `recall` and take their mean.
2. Repeat 1,000 times. Store the 1,000 means in `boot_means`.
3. The 95% interval runs from the 2.5th to the 97.5th percentile of
   `boot_means`. Print it.

**Expected.** About 0.95 to 0.97.

In [ ]:
# Your code here.

# Part B — What the measurement shows

## B1 — Keyword search ✏️ Write this yourself

The cell below runs keyword search on the same 1,000 queries. It compares
questions by the words they share, with TF-IDF weights, and keeps the top 10.
It uses no trained model. The result is `found_kw`, in the same form as
`found`.

Run it first. It takes one to two minutes.

In [ ]:
tfidf = TfidfVectorizer()
T = tfidf.fit_transform(texts)
TQ = tfidf.transform([query_text[q] for q in queries])
found_kw = []
for start in range(0, len(queries), 50):
    scores = (TQ[start:start + 50] @ T.T).toarray()
    found_kw += [ids[np.argsort(-row)[:10]].tolist() for row in scores]
print("Done.")

**Goal.** Compare keyword search with the embeddings.

1. Compute recall at 10 for `found_kw` with your function from A5. Store it in
   `recall_kw` and print its mean.
2. Count the queries where the embeddings have the higher recall, and the
   queries where keyword search has the higher recall.
3. Print the text of one query where the embeddings found a marked duplicate
   and keyword search found none.

**Expected.** Keyword search: about 0.86. The embeddings are higher for
152 queries, and keyword search for 20.

In [ ]:
# Your code here.

**Answer here.** Look at your example query. Why does keyword search miss the
marked duplicate?

*Answer here.*

## B2 — Shorter vectors ✏️ Write this yourself

**Goal.** Measure what the search loses when each question keeps only its
first k principal components.

`PCA` stands for *principal component analysis*, the method of the slides. It
centres the vectors itself, so you do not need to subtract the average.

1. Fit `PCA(n_components=50)` on 200,000 questions drawn at random from `X`.
   Use `np.random.default_rng(1)` for the draw.
2. For k = 50 and k = 10: transform `X` and `Q` with the fitted PCA, keep the
   first k columns, and divide each row by its length so that it has length 1
   again.
3. Run `top10` on the shortened vectors, compute recall at 10, and print it
   for each k.

**Expected.** About 0.89 with 50 numbers and about 0.51 with 10 numbers.

In [ ]:
# Your code here.

**Answer here.** 50 components keep about 45% of the total variation. Why might recall drop much less than that?

*Answer here.*

## B3 — Speed against recall ✏️ Write this yourself

The cell below builds an approximate index. It divides the vectors into 1,024
groups and, for each query, searches only the questions in the nearest groups.
`index.nprobe` sets how many groups are searched.

Run it first. It takes about a minute.

In [ ]:
faiss.omp_set_num_threads(1)          # one processor thread, as on the slides
quantizer = faiss.IndexFlatIP(384)
index = faiss.IndexIVFFlat(quantizer, 384, 1024, faiss.METRIC_INNER_PRODUCT)
index.train(X[np.random.default_rng(7).choice(len(X), 100_000, replace=False)])
index.add(X)
print("Index ready.")

**Goal.** Measure recall at 10 and the time per query for 1, 2, 4, 8, 16 and
32 groups searched.

For each number of groups:

1. Set `index.nprobe`.
2. Time `_, I = index.search(Q, 10)` with `time.perf_counter()`, and divide by
   1,000 to get the time per query in milliseconds.
3. Turn `I` into ids with `ids[I]`, and compute recall at 10.

Store the results in two lists, `times` and `recalls`. The plot cell after
yours uses them.

**Expected.** Recall rises from about 0.76 with 1 group to about 0.95 with
32 groups. The times depend on your machine.

In [ ]:
# Your code here.

In [ ]:
plt.plot(times, recalls, "o-")
for g, t, r in zip(groups, times, recalls):
    plt.annotate(str(g), (t, r), textcoords="offset points", xytext=(0, -14), ha="center")
plt.xscale("log")
plt.xlabel("time per query (ms, log scale)")
plt.ylabel("recall at 10")
plt.title("Numbers: groups searched, out of 1,024")
plt.show()

**Answer here.** A website must answer each search in under 1 ms. Which number
of groups would you choose, and what recall do you give up compared with exact
search?

*Answer here.*

## B4 — Missing labels

The cell below takes six of the queries. For each, it prints the questions in
the top 10 that have a cosine similarity above 0.90 but are not marked as
duplicates.

Run it and read the pairs.

In [ ]:
pos = {question_id: i for i, question_id in enumerate(ids)}
examples = ["7676", "14915", "475286", "30597", "234257", "398366"]
for q in examples:
    v, top = Q[queries.index(q)], found[queries.index(q)]
    print(query_text[q])
    for question_id in top:
        cos = float(X[pos[question_id]] @ v)
        if cos > 0.90 and question_id not in marked[q]:
            print(f"  {cos:.2f}  {texts[pos[question_id]]}")
    print()

**Answer here.** Pick three pairs. For each, say whether the two questions ask
the same thing. What does this mean for the recall at 10 of A5?

*Answer here.*

## B5 — One limitation

The model was trained on more than 1 billion examples of related sentences.
They include 103,663 Quora triplets. Each triplet is a question, a marked
duplicate, and a question that is not a duplicate.

**Answer here.** How could this affect the recall at 10 of A5? Describe a test
that would show how well the model works on questions it has never seen.

*Answer here.*

---
**Before you submit.** Restart the kernel and run every cell from top to
bottom. A2 is fast the second time, because the vectors are saved.